In [ ]:
#2
#perl, version 22
#нужно: augustus.whole.gff (аннотация), GCA_001949185.1_Rvar_4.0_genomic.fna (геном)
mamba install -c bioconda perl
perl getAnnoFasta.pl augustus.whole.gff --seqfile=GCA_001949185.1_Rvar_4.0_genomic.fna #у неё какой-то странный output 'Read in 200 sequence(s) from GCA_001949185.1_Rvar_4.0_genomic.fna.', но правильный выходной файл появляется

grep ">" augustus.whole.aa | wc -l
#выход - augustus.whole.aa - белки все - 16435 штук

peptides.fa - список пептидов, компонентов белков, связанных с ДНК по итогам эксперементальной проверки  
Цель - узнать, каким белкам из генома тихоходки соответствуют эти пептиды

In [ ]:
#создайте локальную базу данных из вашего файла fasta с белками и выполните поиск, используя файл с последовательностью пептидов в качестве запроса
mamba install -c bioconda samtools blast
#samtools - 1.23.1
#blast
#создать базу данных
makeblastdb -in augustus.whole.aa -dbtype prot -out database_tard_prot
#поиск
blastp -db database_tard_prot -query peptides.fa -outfmt 6 -out DNA_proteins
#итог - файл DNA_proteins - последовательности белков, связывающихся с ДНК у тихоходок

#извлечь нужные белки из исходного файла
samtools faidx augustus.whole.aa #индекcирование
cut -f2 DNA_proteins | sort | uniq > ids.txt #извлечение
xargs samtools faidx augustus.whole.aa < ids.txt > selected_prot.fa

grep ">" selected_prot.fa | wc -l
#получилось 34 белка

получаем белки, связаннные с ДНК у тихоходок - selected_prot.fa  

##### Следующий этап - определяем расположение этих белков в клетке:  
1) WoLF PSORT - по наличию N-концевого пептида внеклеточного расположения. Нужно отправить на сервер список белков selected_prot.fa, выбрать организм
https://wolfpsort.hgc.jp/
Там будут вероятности каждой локализации
2) TargetP Server - по наличию любой из N-концевых препоследовательностей: хлоропластного транзитного пептида (cTP), митохондриального целевого пептида (mTP) или сигнального пептида секреторного пути (SP). Тоже оптравить список белков и выбрать организм
https://services.healthtech.dtu.dk/service.php?TargetP-2.  
Вывод:  
OTHER — “никуда не направляется” (нет сигнала)  
SP (signal peptide) — сигнал секреции (в секреторный путь)  
mTP (mitochondrial targeting peptide) — сигнал в митохондрию  

In [ ]:
#поиск белковых последовательностей blast
#получение базы данных
wget https://ftp.ncbi.nlm.nih.gov/blast/db/swissprot.tar.gz
tar -xzf swissprot.tar.gz
#запуск
blastp \
  -db ~/blastdb/swissprot \
  -query Polina/selected_prot.fa \
  -out blast_swissprot_results.txt \
  -outfmt "6 qseqid sseqid evalue pident qcovs stitle" \
  -evalue 1e-5 \
  -num_threads 8 \
  -max_target_seqs 1


##### прогнозирование функции HMMER
веб версия https://www.ebi.ac.uk/Tools/hmmer/  
инструмент - hmmscan  
база данных - Pfam  